# Qwen Baseline Fine-Tuning (Colab) — Finetune-RAG

This notebook fine-tunes a baseline Qwen instruct model using `pints-ai/Finetune-RAG` with QLoRA, then saves/pushes the adapter to Hugging Face.

## Runtime expectations
- Recommended: **Google Colab GPU** (T4, L4, A100)
- Python 3.10+
- Hugging Face token with write access (for pushing model artifacts)

## Outputs
- LoRA adapter checkpoint
- Tokenizer files
- Optional push to Hugging Face Hub

In [1]:
print('hello world')

hello world


In [3]:
import torch, platform, os
print("Python:", platform.python_version())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("CUDA version (torch):", torch.version.cuda)
else:
    print("No GPU attached")

Python: 3.9.6
CUDA available: False
No GPU attached


In [ ]:
!pip -q install -U -r training_requirements.txt

zsh:1: command not found: pip


In [3]:
import os
import random
from dataclasses import dataclass

import torch
from datasets import Dataset, load_dataset
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

ModuleNotFoundError: No module named 'datasets'

In [ ]:
# Colab/runtime package setup via requirements file
# Better for reproducibility and easy version control.



In [ ]:
import os
import random
from dataclasses import dataclass

import torch
from datasets import Dataset, load_dataset
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Hugging Face authentication
# Option 1 (recommended in Colab): set token in secret named HF_TOKEN
# Option 2: uncomment login() and paste token interactively

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in from HF_TOKEN env var")
else:
    print("HF_TOKEN not found in environment. Run `login()` manually if needed.")
    # login()

In [ ]:
@dataclass
class Config:
    base_model: str = "Qwen/Qwen2.5-7B-Instruct"  # baseline Qwen to fine-tune
    dataset_name: str = "pints-ai/Finetune-RAG"
    output_dir: str = "./qwen2_5_7b_finetunerag_lora"
    push_repo_id: str = "<your-username>/qwen2.5-7b-finetunerag-lora"

    max_seq_len: int = 2048
    test_size: float = 0.1

    epochs: float = 2.0
    learning_rate: float = 2e-4
    train_batch_size: int = 2
    eval_batch_size: int = 2
    grad_accum_steps: int = 8
    warmup_ratio: float = 0.03
    weight_decay: float = 0.0

    lora_r: int = 64
    lora_alpha: int = 16
    lora_dropout: float = 0.05

    use_4bit: bool = True
    include_distractors: bool = True

cfg = Config()
cfg

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful retrieval-augmented assistant. "
    "Answer only using retrieved context. "
    "If the answer is not supported by context, say so clearly."
)


def safe_text(x):
    return "" if x is None else str(x).strip()


def build_context(row, include_distractors=True):
    parts = []
    before = safe_text(row.get("content_before"))
    main = safe_text(row.get("content"))
    after = safe_text(row.get("content_after"))

    if before:
        parts.append(f"[Context before]\n{before}")
    if main:
        parts.append(f"[Primary evidence]\n{main}")
    if after:
        parts.append(f"[Context after]\n{after}")

    if include_distractors:
        d1 = safe_text(row.get("fictitious_content1"))
        d2 = safe_text(row.get("fictitious_content2"))
        if d1:
            parts.append(f"[Additional retrieved snippet A]\n{d1}")
        if d2:
            parts.append(f"[Additional retrieved snippet B]\n{d2}")

    return "\n\n".join(parts).strip()


def to_messages(row, include_distractors=True):
    question = safe_text(row.get("question"))
    answer = safe_text(row.get("answer"))
    context = build_context(row, include_distractors=include_distractors)

    user = (
        "Question:\n"
        f"{question}\n\n"
        "Retrieved context:\n"
        f"{context}\n\n"
        "Instructions:\n"
        "- Use only retrieved context.\n"
        "- If not supported, say it is not in context.\n"
        "- Do not hallucinate facts."
    )

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user},
        {"role": "assistant", "content": answer},
    ]

In [ ]:
# Load dataset + tokenizer
raw = load_dataset(cfg.dataset_name)
train_raw = raw["train"]

print(f"Raw rows: {len(train_raw)}")

# Train / eval split
split = train_raw.train_test_split(test_size=cfg.test_size, seed=SEED)
train_split = split["train"]
eval_split = split["test"]

print(f"Train: {len(train_split)} | Eval: {len(eval_split)}")

tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def to_text(row):
    messages = to_messages(row, include_distractors=cfg.include_distractors)
    if tokenizer.chat_template:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    else:
        text = (
            f"<|system|>\n{messages[0]['content']}\n"
            f"<|user|>\n{messages[1]['content']}\n"
            f"<|assistant|>\n{messages[2]['content']}"
        )
    return {"text": text}

train_text = train_split.map(to_text, remove_columns=train_split.column_names)
eval_text = eval_split.map(to_text, remove_columns=eval_split.column_names)

print(train_text[0]["text"][:1000])

In [ ]:
# Tokenize

def tokenize_batch(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=cfg.max_seq_len,
    )
    out["labels"] = out["input_ids"].copy()
    return out

train_tok = train_text.map(tokenize_batch, batched=True)
eval_tok = eval_text.map(tokenize_batch, batched=True)

train_tok = train_tok.remove_columns([c for c in train_tok.column_names if c not in {"input_ids", "attention_mask", "labels"}])
eval_tok = eval_tok.remove_columns([c for c in eval_tok.column_names if c not in {"input_ids", "attention_mask", "labels"}])

train_tok.set_format(type="torch")
eval_tok.set_format(type="torch")

print(train_tok[0].keys())

In [ ]:
# Load baseline Qwen model with optional 4-bit quantization
if cfg.use_4bit and not torch.cuda.is_available():
    raise RuntimeError("use_4bit=True requires CUDA. Set cfg.use_4bit=False on CPU/MPS.")

model_kwargs = {"trust_remote_code": True}

if cfg.use_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    )
    model_kwargs["quantization_config"] = bnb_config
    model_kwargs["device_map"] = "auto"
else:
    model_kwargs["torch_dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model = AutoModelForCausalLM.from_pretrained(cfg.base_model, **model_kwargs)
model.config.use_cache = False

if cfg.use_4bit:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [4]:
# Trainer setup + training
bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16 = torch.cuda.is_available() and not bf16

training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.epochs,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.train_batch_size,
    per_device_eval_batch_size=cfg.eval_batch_size,
    gradient_accumulation_steps=cfg.grad_accum_steps,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    eval_strategy="steps",
    save_strategy="steps",
    lr_scheduler_type="cosine",
    bf16=bf16,
    fp16=fp16,
    gradient_checkpointing=True,
    report_to="none",
    dataloader_pin_memory=False,
)

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=collator,
)

trainer.train(resume_from_checkpoint=True)

NameError: name 'TrainingArguments' is not defined

In [ ]:
!nvidia-smi -q -d ECC

In [ ]:
# Save LoRA adapter + tokenizer
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)

print(f"Saved adapter + tokenizer to: {cfg.output_dir}")

In [ ]:
# Optional: push adapter to Hugging Face Hub
# Make sure cfg.push_repo_id is set to your namespace/repo

if "<your-username>" in cfg.push_repo_id:
    print("Set cfg.push_repo_id before pushing, e.g. 'myname/qwen2.5-7b-finetunerag-lora'")
else:
    model.push_to_hub(cfg.push_repo_id)
    tokenizer.push_to_hub(cfg.push_repo_id)
    print(f"Pushed to https://huggingface.co/{cfg.push_repo_id}")

## Load this fine-tuned adapter in your repo eval flow

After pushing, use your adapter repo ID in your evaluation/inference path.

Example faithfulness re-eval command in `llm_poc/`:

```bash
../../.venv/bin/python eval/run_faithfulness_eval.py \
  --triples eval/evaluation_datasets/triples/pdf_retrieval_triples.jsonl \
  --answer-model <your-hf-finetuned-model-id-or-endpoint> \
  --run-name faithfulness_qwen_sft_candidate
```

Then compare against existing runs in:
- `eval/results/llm_faithfulness_metrics/comparison.csv`

## Colab Evaluation (Local Inference): Faithfulness + Abstention

Use this section after uploading the notebook to Colab.

What this gives you:
- Load your merged fine-tuned model in Colab
- Run faithfulness evaluation on `pdf_retrieval_triples.jsonl`
- Run abstention evaluation on variable + paper abstention datasets
- Save JSON + CSV outputs to Drive or `/content`

Notes:
- This runs locally in the Colab runtime (no HF Inference provider needed).
- For the 7B merged model, prefer an L4/A100 runtime.
- You can set `MAX_QUERIES` for quick smoke tests first.

In [ ]:
# Colab eval dependencies
!pip -q install -U transformers datasets accelerate bitsandbytes sentencepiece pandas

In [ ]:
# Common setup for local Colab evaluation
import json
import math
import random
import re
import time
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ---- Paths: update for your Drive layout if needed ----
# Example after mounting Drive:
# from google.colab import drive
# drive.mount('/content/drive')

MODEL_ID = "dizza01/qwen2.5-7b-finetunerag-merged"

TRIPLES_PATH = "/content/drive/MyDrive/BiB/eval/evaluation_datasets/triples/pdf_retrieval_triples.jsonl"
VAR_ABS_PATH = "/content/drive/MyDrive/BiB/eval/evaluation_datasets/variable_abstention/abstention_benchmark.jsonl"
PAPER_ABS_PATH = "/content/drive/MyDrive/BiB/eval/evaluation_datasets/paper_abstention/paper_abstention_benchmark.jsonl"

OUT_DIR = "/content/drive/MyDrive/BiB/eval/results/colab_local_eval"

USE_4BIT = True
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


def _load_jsonl(path: str):
    rows = []
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


def _extract_json(text: str):
    text = (text or "").strip()
    if text.startswith("```"):
        lines = text.splitlines()
        lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        text = "\n".join(lines).strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not m:
            raise
        return json.loads(m.group(0))


def _tokenize_words(text: str):
    return re.findall(r"[a-z0-9]+", (text or "").lower())


def token_f1(pred: str, gold: str) -> float:
    p = _tokenize_words(pred)
    g = _tokenize_words(gold)
    if not p or not g:
        return 0.0

    pc = {}
    for t in p:
        pc[t] = pc.get(t, 0) + 1

    gc = {}
    for t in g:
        gc[t] = gc.get(t, 0) + 1

    overlap = 0
    for t, c in pc.items():
        overlap += min(c, gc.get(t, 0))

    prec = overlap / len(p)
    rec = overlap / len(g)
    return 0.0 if (prec + rec) == 0 else (2 * prec * rec / (prec + rec))


_loaded = {"tokenizer": None, "model": None}


def load_eval_model(model_id: str = MODEL_ID, use_4bit: bool = USE_4BIT):
    if _loaded["model"] is not None and _loaded["tokenizer"] is not None:
        return _loaded["tokenizer"], _loaded["model"]

    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if torch.cuda.is_available():
        compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    else:
        compute_dtype = torch.float16 if torch.backends.mps.is_available() else torch.float32

    model_kwargs = {
        "torch_dtype": compute_dtype,
        "low_cpu_mem_usage": True,
    }

    if torch.cuda.is_available() and use_4bit:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=compute_dtype,
        )
        model_kwargs["device_map"] = "auto"
    elif torch.cuda.is_available():
        model_kwargs["device_map"] = "auto"

    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)

    if not torch.cuda.is_available() and torch.backends.mps.is_available():
        model = model.to("mps")

    _loaded["tokenizer"] = tokenizer
    _loaded["model"] = model
    return tokenizer, model


def generate_chat(messages, max_new_tokens=256, temperature=0.0):
    tokenizer, model = load_eval_model()

    if tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        prompt = "\n".join([f"{m['role']}: {m['content']}" for m in messages]) + "\nassistant:"

    inputs = tokenizer(prompt, return_tensors="pt")

    if not hasattr(model, "hf_device_map"):
        device = next(model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": temperature > 0,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0:
        gen_kwargs["temperature"] = temperature

    with torch.inference_mode():
        out = model.generate(**inputs, **gen_kwargs)

    new_tokens = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("Setup complete.")
print("Output directory:", OUT_DIR)

In [ ]:
# Faithfulness evaluation (local, Colab-compatible)
FAITHFULNESS_JUDGE_SYSTEM = """You are a strict faithfulness judge for RAG answers.
Use ONLY the provided context.
Return JSON only with keys:
{
  \"faithful\": true/false,
  \"supported_claims\": <int>,
  \"contradicted_claims\": <int>,
  \"not_found_claims\": <int>,
  \"notes\": \"short note\"
}
Set faithful=true only when contradicted_claims==0 and not_found_claims==0.
"""


def _safe_int(x, default=0):
    try:
        return int(x)
    except Exception:
        return default


def _build_eval_context(row):
    # Use direct context from triples file so no local Chroma index is required.
    return (row.get("source_chunk") or "").strip()


def run_faithfulness_eval_colab(
    triples_path=TRIPLES_PATH,
    max_queries=50,
    answer_max_new_tokens=220,
    judge_max_new_tokens=220,
    out_dir=OUT_DIR,
    run_name="faithfulness_colab_local",
):
    rows = _load_jsonl(triples_path)
    if max_queries and max_queries > 0:
        rows = rows[:max_queries]

    results = []
    start = time.perf_counter()

    for i, row in enumerate(rows, start=1):
        q = (row.get("question") or "").strip()
        ref = (row.get("answer") or "").strip()
        ctx = _build_eval_context(row)
        qid = row.get("query_id", f"q_{i}")

        if not q or not ctx:
            continue

        t0 = time.perf_counter()
        answer = generate_chat(
            [
                {
                    "role": "system",
                    "content": "You are a careful retrieval-augmented assistant. Use only the retrieved context.",
                },
                {
                    "role": "user",
                    "content": (
                        f"Retrieved context:\n\n{ctx}\n\n"
                        f"Question: {q}\n\n"
                        "Instructions: Use only the context. If unsupported, say it is not in context."
                    ),
                },
            ],
            max_new_tokens=answer_max_new_tokens,
            temperature=0.0,
        )
        answer_ms = (time.perf_counter() - t0) * 1000.0

        t1 = time.perf_counter()
        judge_raw = generate_chat(
            [
                {"role": "system", "content": FAITHFULNESS_JUDGE_SYSTEM},
                {
                    "role": "user",
                    "content": (
                        f"Question:\n{q}\n\n"
                        f"Context:\n{ctx}\n\n"
                        f"Answer:\n{answer}"
                    ),
                },
            ],
            max_new_tokens=judge_max_new_tokens,
            temperature=0.0,
        )
        judge_ms = (time.perf_counter() - t1) * 1000.0

        faithful = False
        supported = 0
        contradicted = 0
        not_found = 0
        notes = ""
        parse_error = ""

        try:
            parsed = _extract_json(judge_raw)
            faithful = bool(parsed.get("faithful", False))
            supported = _safe_int(parsed.get("supported_claims", 0))
            contradicted = _safe_int(parsed.get("contradicted_claims", 0))
            not_found = _safe_int(parsed.get("not_found_claims", 0))
            notes = str(parsed.get("notes", "")).strip()
            faithful = bool((contradicted == 0) and (not_found == 0) and faithful)
        except Exception as e:
            parse_error = str(e)

        f1 = token_f1(answer, ref)
        results.append(
            {
                "query_id": qid,
                "question": q,
                "reference_answer": ref,
                "rag_answer": answer,
                "answer_token_f1": f1,
                "faithful": faithful,
                "supported_claims": supported,
                "contradicted_claims": contradicted,
                "not_found_claims": not_found,
                "judge_notes": notes,
                "judge_parse_error": parse_error,
                "latency_answer_ms": answer_ms,
                "latency_judge_ms": judge_ms,
            }
        )
        print(f"[{len(results):>3}/{len(rows)}] {qid} | faithful={faithful} | f1={f1:.3f}")

    n = len(results)
    total_claims = sum(r["supported_claims"] + r["contradicted_claims"] + r["not_found_claims"] for r in results)
    total_unfaithful = sum(r["contradicted_claims"] + r["not_found_claims"] for r in results)

    summary = {
        "n_queries": n,
        "query_faithfulness_rate": (sum(1 for r in results if r["faithful"]) / n) if n else 0.0,
        "unfaithful_claim_rate": (total_unfaithful / total_claims) if total_claims else 0.0,
        "avg_answer_token_f1": (sum(r["answer_token_f1"] for r in results) / n) if n else 0.0,
        "avg_answer_latency_ms": (sum(r["latency_answer_ms"] for r in results) / n) if n else 0.0,
        "avg_judge_latency_ms": (sum(r["latency_judge_ms"] for r in results) / n) if n else 0.0,
        "elapsed_seconds": time.perf_counter() - start,
    }

    out_json = Path(out_dir) / f"{run_name}.json"
    out_csv = Path(out_dir) / f"{run_name}.csv"

    with out_json.open("w", encoding="utf-8") as fh:
        json.dump({"summary": summary, "per_query": results}, fh, ensure_ascii=False, indent=2)

    pd.DataFrame(results).to_csv(out_csv, index=False)

    print("\nFaithfulness summary:")
    print(json.dumps(summary, indent=2))
    print("Saved:", out_json)
    print("Saved:", out_csv)
    return summary, results

In [ ]:
# Abstention evaluation (local, Colab-compatible)
ABSTAIN_PATTERNS = [
    r"\binsufficient context\b",
    r"\bnot in (the )?context\b",
    r"\bi don't have enough information\b",
    r"\bcannot determine\b",
    r"\bcan('?)t determine\b",
    r"\bnot enough information\b",
    r"\bunable to answer\b",
]


def _detect_abstention(answer: str) -> bool:
    text = (answer or "").lower()
    return any(re.search(p, text) for p in ABSTAIN_PATTERNS)


def run_abstention_eval_colab(
    variable_dataset_path=VAR_ABS_PATH,
    paper_dataset_path=PAPER_ABS_PATH,
    max_queries_per_slice=150,
    answer_max_new_tokens=120,
    out_dir=OUT_DIR,
    run_name="abstention_colab_local",
):
    var_rows = _load_jsonl(variable_dataset_path)
    pap_rows = _load_jsonl(paper_dataset_path)

    if max_queries_per_slice and max_queries_per_slice > 0:
        var_rows = var_rows[:max_queries_per_slice]
        pap_rows = pap_rows[:max_queries_per_slice]

    rows = []
    for r in var_rows:
        x = dict(r)
        x["slice"] = "variable_abstention"
        rows.append(x)
    for r in pap_rows:
        x = dict(r)
        x["slice"] = "paper_abstention"
        rows.append(x)

    results = []
    for i, row in enumerate(rows, start=1):
        q = (row.get("question") or "").strip()
        qid = row.get("query_id", f"abs_{i}")
        should_abstain = bool(row.get("should_abstain", False))

        # In Colab-only mode, we evaluate abstention behavior without local Chroma retrieval.
        answer = generate_chat(
            [
                {
                    "role": "system",
                    "content": (
                        "You are a careful research assistant. "
                        "If the answer cannot be verified from provided evidence, say: 'Insufficient context.'"
                    ),
                },
                {
                    "role": "user",
                    "content": (
                        f"Question: {q}\n\n"
                        "Retrieved context: [none provided]\n\n"
                        "If unsupported, abstain clearly."
                    ),
                },
            ],
            max_new_tokens=answer_max_new_tokens,
            temperature=0.0,
        )

        pred_should_abstain = _detect_abstention(answer)

        results.append(
            {
                "query_id": qid,
                "slice": row.get("slice", ""),
                "question": q,
                "should_abstain": should_abstain,
                "pred_should_abstain": pred_should_abstain,
                "correct": bool(should_abstain == pred_should_abstain),
                "reason": row.get("reason", ""),
                "generation_type": row.get("generation_type", ""),
                "source_scope": row.get("source_scope", ""),
                "answer": answer,
            }
        )

        print(
            f"[{len(results):>3}/{len(rows)}] {qid} | "
            f"gold_abstain={should_abstain} | pred_abstain={pred_should_abstain}"
        )

    df = pd.DataFrame(results)

    tp = int(((df["should_abstain"] == True) & (df["pred_should_abstain"] == True)).sum())
    tn = int(((df["should_abstain"] == False) & (df["pred_should_abstain"] == False)).sum())
    fp = int(((df["should_abstain"] == False) & (df["pred_should_abstain"] == True)).sum())
    fn = int(((df["should_abstain"] == True) & (df["pred_should_abstain"] == False)).sum())

    n = len(df)
    n_pos = int((df["should_abstain"] == True).sum())
    n_neg = int((df["should_abstain"] == False).sum())

    precision = (tp / (tp + fp)) if (tp + fp) else 0.0
    recall = (tp / n_pos) if n_pos else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

    summary = {
        "n_examples": n,
        "n_should_abstain": n_pos,
        "n_should_answer": n_neg,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "abstention_accuracy": ((tp + tn) / n) if n else 0.0,
        "true_abstain_rate": recall,
        "false_answer_rate": (fn / n_pos) if n_pos else 0.0,
        "false_abstain_rate": (fp / n_neg) if n_neg else 0.0,
        "precision_abstain": precision,
        "recall_abstain": recall,
        "f1_abstain": f1,
    }

    by_slice = []
    for slice_name, part in df.groupby("slice"):
        by_slice.append(
            {
                "slice": slice_name,
                "n_examples": int(len(part)),
                "abstention_accuracy": float((part["correct"] == True).mean()),
            }
        )

    out_json = Path(out_dir) / f"{run_name}.json"
    out_csv = Path(out_dir) / f"{run_name}.csv"

    with out_json.open("w", encoding="utf-8") as fh:
        json.dump(
            {
                "summary": summary,
                "by_slice": by_slice,
                "per_query": results,
            },
            fh,
            ensure_ascii=False,
            indent=2,
        )

    df.to_csv(out_csv, index=False)

    print("\nAbstention summary:")
    print(json.dumps(summary, indent=2))
    print("By slice:", by_slice)
    print("Saved:", out_json)
    print("Saved:", out_csv)
    return summary, df

In [ ]:
# Run faithfulness eval (start with a small smoke test)
faithfulness_summary, faithfulness_rows = run_faithfulness_eval_colab(
    triples_path=TRIPLES_PATH,
    max_queries=30,
    answer_max_new_tokens=220,
    judge_max_new_tokens=220,
    out_dir=OUT_DIR,
    run_name="faithfulness_qwen_finetuned_colab_local_smoke",
)

In [ ]:
# Run abstention eval (start with a small smoke test)
abstention_summary, abstention_df = run_abstention_eval_colab(
    variable_dataset_path=VAR_ABS_PATH,
    paper_dataset_path=PAPER_ABS_PATH,
    max_queries_per_slice=80,
    answer_max_new_tokens=120,
    out_dir=OUT_DIR,
    run_name="abstention_qwen_finetuned_colab_local_smoke",
)

abstention_df.head(5)

### Colab Run Order

1. Run the dependency cell.
2. (Optional) Mount Drive and update `TRIPLES_PATH`, `VAR_ABS_PATH`, `PAPER_ABS_PATH`, `OUT_DIR`.
3. Run the common setup cell (loads helper functions).
4. Run the faithfulness function-definition cell.
5. Run the abstention function-definition cell.
6. Run the faithfulness smoke test cell.
7. Run the abstention smoke test cell.

For full runs, increase:
- `max_queries` in `run_faithfulness_eval_colab` (set to `0` to use all rows)
- `max_queries_per_slice` in `run_abstention_eval_colab` (set to `0` to use all rows)

Outputs are saved as JSON + CSV in `OUT_DIR`.